Propósito: Filtra los tickers propuestos para traerlos filtrados por condiciones.

Usa parametros del indicador RSI de 60min, 1d y 7d.

Este filtro busca capturar movimientos de la siguiente forma: Activo sobrevendido en 1d y 60m; la temporalidad baja indica el movimiento de reversion alcista en conjunto con una temporalidad diaria subiendo. La temporalidad más larga indica la tendencia del activo en largo plazo.

In [1]:
# Instala las librerias requeridas.

%pip install -q pandas-ta yfinance lxml

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 23.2.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
# Importa las librerias requeridas.
import pandas as pd
import requests
import io
import yfinance as yf
import pandas_ta as ta

In [3]:
#Scrapea Wikipedia para acceder a los nombres de los tickers del S&P500.

#1.1 Define la URL y un 'User-Agent' para evitar el bloqueo.
url = 'https://en.wikipedia.org/wiki/List_of_S%26P_500_companies'
header = {
  "User-Agent": "Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/50.0.2661.75 Safari/537.36",
  "X-Requested-With": "XMLHttpRequest"
}

#1.2 Realiza la petición con el header de seguridad
r = requests.get(url, headers=header)

#1.3 Lee la tabla desde el texto de la respuesta, usando io.StringIO para forzar la interpretación como string HTML
tables = pd.read_html(io.StringIO(r.text))
df_spy = tables[0]

#1.4 Extrae la lista de Tickers
tickers_a_analizar = [t.replace('.', '-') for t in df_spy['Symbol'].tolist()]

#1.5 Añade tickers de interes.

mis_tickers = ['^VIX', 'SPY', 'SPXL' , 'TQQQ' , 'QQQ' , 'URA' , 'XLI' , 'XLF' , 'XLRE' , 'XLB' , 'XLY' , 'XLE', 'XLV' ,
                'XBI' , 'XLC' , 'BTC-USD' , 'DIA']

tickers_a_analizar.extend(mis_tickers)

if tickers_a_analizar:
  print(f"Se han cargado {len(tickers_a_analizar)} tickers...")

Se han cargado 520 tickers...


In [13]:
# Índice de volatilidad VIX.

tickers_a_analizar

print(tickers_a_analizar)

['MMM', 'AOS', 'ABT', 'ABBV', 'ACN', 'ADBE', 'AMD', 'AES', 'AFL', 'A', 'APD', 'ABNB', 'AKAM', 'ALB', 'ARE', 'ALGN', 'ALLE', 'LNT', 'ALL', 'GOOGL', 'GOOG', 'MO', 'AMZN', 'AMCR', 'AEE', 'AEP', 'AXP', 'AIG', 'AMT', 'AWK', 'AMP', 'AME', 'AMGN', 'APH', 'ADI', 'AON', 'APA', 'APO', 'AAPL', 'AMAT', 'APP', 'APTV', 'ACGL', 'ADM', 'ARES', 'ANET', 'AJG', 'AIZ', 'T', 'ATO', 'ADSK', 'ADP', 'AZO', 'AVY', 'AXON', 'BKR', 'BALL', 'BAC', 'BAX', 'BDX', 'BRK-B', 'BBY', 'TECH', 'BIIB', 'BLK', 'BX', 'XYZ', 'BE', 'BNY', 'BA', 'BKNG', 'BSX', 'BMY', 'AVGO', 'BR', 'BRO', 'BF-B', 'BG', 'BXP', 'CHRW', 'CDNS', 'CPT', 'COF', 'CAH', 'CCL', 'CARR', 'CVNA', 'CASY', 'CAT', 'CBOE', 'CBRE', 'CDW', 'COR', 'CNC', 'CNP', 'CF', 'CRL', 'SCHW', 'CHTR', 'CVX', 'CMG', 'CB', 'CHD', 'CIEN', 'CI', 'CINF', 'CTAS', 'CSCO', 'C', 'CFG', 'CLX', 'CME', 'CMS', 'KO', 'CTSH', 'COHR', 'COIN', 'CL', 'CMCSA', 'FIX', 'COP', 'ED', 'STZ', 'CEG', 'COO', 'CPRT', 'GLW', 'CPAY', 'CSGP', 'COST', 'CRH', 'CRWD', 'CCI', 'CSX', 'CMI', 'CVS', 'DHR', 'DRI', 

In [4]:
#DESCARGA DE DATOS TEMPORALIDAD 60 MINUTOS.

tickers_limpios = tickers_a_analizar
data_60m = yf.download(tickers_limpios, period="2mo", interval="60m", group_by='ticker', progress=False)

In [5]:
#ANALISIS TEMPORALIDAD 60MIN.

empresas_60min = []

for ticker in tickers_limpios:
    try:
        df = data_60m[ticker].dropna()
        if len(df) < 50: continue

        # Cálculos Técnicos
        df.ta.rsi(length=10, append=True)
        df['RSI_MA'] = ta.sma(df['RSI_10'], length=14)
        df.ta.macd(fast=12, slow=26, signal=9, append=True)

        # Variables para validación.
        rsi_act = df['RSI_10'].iloc[-1]
        rsi_ant = df['RSI_10'].iloc[-2]
        macd_act = df['MACD_12_26_9'].iloc[-1]
        sig_act = df['MACDs_12_26_9'].iloc[-1]

        # Lógica de Filtrado 60min
        if (rsi_act <= 35 
            and rsi_act >= rsi_ant
                ):
            
            empresas_60min.append({
                'Ticker': ticker,
                'RSI actual': round(rsi_act, 1),
                'RSI anterior': round(rsi_ant, 1),
                #Añade el MACD.
                'MACD': round(macd_act, 3)
                
            })
    except Exception as e:
      print(f"Error en {ticker}: {e}")

print("\n" + "="*95)
if empresas_60min:
    df_final = pd.DataFrame(empresas_60min)
    print(df_final.sort_values(by='RSI actual').to_string(index=False))
else:
    print("Sin señales detectadas.")


Ticker  RSI actual  RSI anterior    MACD
  AMAT        18.7          12.9 -11.916
  TQQQ        28.6          26.2  -1.497
   QQQ        28.9          26.6  -4.805
  SNPS        30.3          15.9  -8.732
  INTC        31.7          19.8  -2.104
   FCX        31.9          22.4  -1.305
   TER        32.4          26.1  -9.149
  NVDA        33.7          29.2  -1.621
  ANET        34.8          26.6  -3.585


In [6]:
# DESCARGA DATOS DIARIOS
data_1d = yf.download(tickers_limpios, period="2mo", interval="1d", group_by='ticker', progress=False)

In [7]:
# ANÁLISIS DIARIO

empresas_diario = []

for ticker in tickers_limpios:
    try:
        df = data_1d[ticker].dropna()
        if len(df) < 30: continue

        # Cálculos Técnicos.
        df.ta.rsi(length=14, append=True)
        df.ta.macd(fast=12, slow=26, signal=9, append=True)

        # Variables de validación
        rsi_ant = df['RSI_14'].iloc[-2]
        rsi_act = df['RSI_14'].iloc[-1]
        macd_act = df['MACD_12_26_9'].iloc[-1]
        sig_act = df['MACDs_12_26_9'].iloc[-1]
        

        precio_act = df['Close'].iloc[-1]

        # LÓGICA DE FILTRADO DIARIO
        estuvo_sobrevendido = rsi_act <= 35 or rsi_ant <= 35
        dejo_de_caer = rsi_act >= rsi_ant

        if dejo_de_caer:
            
            empresas_diario.append({
                'Ticker': ticker,
                #'Prec diario': round(precio_act, 2),
                'RSI actual': round(rsi_act, 1),
                'RSI anterior': round(rsi_ant, 1),
                'MACD': round(macd_act, 3),
                #'MACD_SIGNAL': round(sig_act , 3)
            })
    except:
        continue

print("\n" + "="*95)


print("PAPELES DETECTADOS EN DIARIO:")
if empresas_diario:
    df_final = pd.DataFrame(empresas_diario)
    print(df_final.sort_values(by='RSI actual').to_string(index=False))
else:
    print("Sin señales diarias.")


PAPELES DETECTADOS EN DIARIO:
 Ticker  RSI actual  RSI anterior     MACD
    ALB        19.9          18.4   -6.892
   AMAT        24.6          23.9  -22.818
   DXCM        31.1          29.3   -1.649
   SNDK        32.2          29.4  -89.190
   INTC        32.4          30.4   -4.162
    GLW        33.5          33.4   -7.087
    TER        33.8          32.6  -14.946
   COHR        34.2          32.0  -20.545
   ANET        34.8          33.2   -6.177
   AVGO        36.1          34.4   -7.928
    FCX        36.1          34.3   -1.677
    JBL        37.2          36.3   -8.357
   TQQQ        37.3          36.7   -1.555
   AKAM        38.1          36.7   -4.755
    QQQ        38.8          37.9   -3.802
   LITE        39.9          37.2  -41.684
   DELL        40.1          39.5  -17.560
    PLD        40.2          39.7   -2.166
   ORCL        40.9          39.8   -4.952
    BKR        40.9          34.7   -1.218
     MU        41.1          36.2  -31.149
     BG        41.6    

In [8]:
#TICKERS REPETIDOS EN AMBAS TEMPORALIDADES.

tickers_60 = [e['Ticker'] for e in empresas_60min]
ticker_1d = [e['Ticker'] for e in empresas_diario]
tickers_comunes = list(set(tickers_60) & set(ticker_1d)) # Interseccion de los elementos en la lista.

print(tickers_comunes)


['QQQ', 'SNPS', 'TER', 'TQQQ', 'NVDA', 'FCX', 'AMAT', 'ANET', 'INTC']


In [9]:
# DESCARGA DATOS SEMANAL.
data_7d = yf.download(tickers_limpios, period="1y", interval="1wk", group_by='ticker', progress=False)

In [10]:
# ANÁLISIS SEMANAL

empresas_semanal = []

for ticker in tickers_limpios:
    try:
        df = data_7d[ticker].dropna()
        if len(df) < 50: continue

        # Cálculos Técnicos.
        df.ta.rsi(length=10, append=True)
        df.ta.macd(fast=12, slow=26, signal=9, append=True)

        # Variables de validación
        rsi_act = df['RSI_10'].iloc[-1]
        macd_act = df['MACD_12_26_9'].iloc[-1]
        sig_act = df['MACDs_12_26_9'].iloc[-1]
        rsi_ant = df['RSI_10'].iloc[-2]

        # Lógica de filtrado.
        if rsi_act <= 44 and (rsi_act > rsi_ant):
        
            empresas_semanal.append({
                'Ticker': ticker,
                'RSI actual': round(rsi_act, 1),
                'RSI anterior': round(rsi_ant, 1),
                'MACD': round(macd_act, 3),
            })
    except:
        continue

print("\n" + "="*95)
print("ACTIVOS DETECTADOS EN SEMANAL.")
if empresas_semanal:
    df_final = pd.DataFrame(empresas_semanal)
    print(df_final.sort_values(by='RSI actual').to_string(index=False))
else:
    print("Sin señales semanales.")


ACTIVOS DETECTADOS EN SEMANAL.
Ticker  RSI actual  RSI anterior     MACD
   MGM        16.7          16.1   -2.804
    GS        28.1          27.2  -12.139
   PNR        28.9          24.0   -8.325
    MS        29.5          27.5   -1.119
   PLD        31.2          28.1   -2.022
   CRH        32.1          29.3   -7.306
   LII        36.2          28.8  -39.013
  SKYD        36.3          26.1   -0.661
  LULU        36.6          32.8  -13.503
   UAL        36.6          34.4   -1.680
   TXT        36.8          34.5   -3.777
  AKAM        36.8          32.7   -3.780
   ARE        37.2          32.6   -1.086
   STX        37.2          37.1   27.494
   WDC        37.5          34.6   -5.063
  CARR        38.1          35.9   -2.019
  HBAN        38.3          33.1   -0.301
   APO        38.7          35.5   -3.011
  FICO        38.7          35.1 -146.300
   EIX        39.0          35.3   -3.226
   LHX        39.1          36.9  -17.049
  CHRW        39.2          38.7   -7.841
  

In [11]:
#TICKERS REPETIDOS EN TRIPLE PANTALLA.

tickers_60 = [e['Ticker'] for e in empresas_60min]
tickers_1d = [e['Ticker'] for e in empresas_diario]
#tickers_7d = [e['Ticker'] for e in empresas_semanal]

tickers_comunes = list(set(tickers_60) & set(tickers_1d) 
                       #& set(tickers_7d)
                       )

if tickers_comunes:
    print(tickers_comunes)

['QQQ', 'SNPS', 'TER', 'TQQQ', 'NVDA', 'FCX', 'AMAT', 'ANET', 'INTC']
